In [1]:
import pandas as pd
from elasticsearch import Elasticsearch
from datetime import datetime

In [2]:
es = Elasticsearch(['http://ingest-elk-01:9200'])

# get all cowrie sessions last 4 weeks
q = {
    'query': {
        'bool': {
            'must': [
                {'term': {'eventid.keyword': 'cowrie.session.connect'}},
                {'range': {'@timestamp': {'gte': 'now-28d'}}}
            ]
        }
    },
    'size': 10000,
    '_source': ['@timestamp', 'src_ip', 'sensor', 'session']
}
resp = es.search(index='cowrie-*', body=q)
df = pd.DataFrame([h['_source'] for h in resp['hits']['hits']])

In [3]:
print(df.shape)

(8412, 4)


In [2]:
# wait i need to rerun this
print(df.columns)

Index(['@timestamp', 'src_ip', 'sensor', 'session'], dtype='object')


In [3]:
df['@timestamp'] = pd.to_datetime(df['@timestamp'])
df['week'] = df['@timestamp'].dt.isocalendar().week

In [4]:
# sessions per sensor per week
# sarah said to use .keyword for ES terms agg but since we already have the df
# we can just groupby here
weekly = df.groupby(['sensor', 'week']).size()
print(weekly)

sensor          week
sensor-apac-01  10      487
                11      512
                12      623
                13      401
sensor-us-01    10      634
                11      589
                12      712
                13      498
dtype: int64


In [5]:
# i wanted unique IPs not sessions, redo
for sensor in df['sensor'].unique():
    mask = df['sensor'] == sensor
    print(f'{sensor}    {df[mask]["src_ip"].nunique()}')

sensor-apac-01    2023
sensor-us-01      2433


In [6]:
# daily breakdown for us sensor
us = df[df['sensor'] == 'sensor-us-01']
us['day'] = us['@timestamp'].dt.date
us.groupby('day')['session'].count()

/tmp/ipykernel_1234/567890.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  us['day'] = us['@timestamp'].dt.date


day
2026-03-01     82
2026-03-02     91
2026-03-03    104
2026-03-04     88
2026-03-05     97
              ... 
2026-03-24    112
2026-03-25     98
2026-03-26    107
2026-03-27     95
2026-03-28     43
Name: session, Length: 28, dtype: int64

In [4]:
# oh wait week 13 is lower because its not complete yet
print('week 13 is partial (only up to today)')

week 13 is partial (only up to today)


In [5]:
# redo without week 13
# sarah said to use .keyword for ES terms queries but we're just doing pandas so doesnt matter here
complete = df[df['week'] != 13]
daily_avg = complete.groupby(['week', 'sensor']).apply(
    lambda g: g.groupby(g['@timestamp'].dt.date)['session'].count().mean()
)
daily_avg

week  sensor        
10    sensor-apac-01     82.3
      sensor-us-01      103.8
11    sensor-apac-01     91.4
      sensor-us-01       98.2
12    sensor-apac-01    106.1
      sensor-us-01      118.7
dtype: float64

In [6]:
# overall avgs for the report
for sensor in ['sensor-us-01', 'sensor-apac-01']:
    mask = complete['sensor'] == sensor
    days = complete[mask]['@timestamp'].dt.date.nunique()
    total = len(complete[mask])
    name = sensor.split('-')[1]
    print(f'{name} avg daily: {total/days:.1f}')

us avg daily: 106.9
apac avg daily: 93.3


In [7]:
# eu sensor
eu = df[df['sensor'] == 'sensor-eu-01']
if len(eu) == 0:
    print('eu-01: no data (offline since week 11)')
else:
    print(f'eu-01: {len(eu)} sessions')

eu-01: no data (offline since week 11)
